# NISAR Pauli Decomposition

**Source script:** `grdl/example/image_processing/sar/nisar_pauli_decomposition.py`

This notebook demonstrates quad-pol SAR Pauli decomposition on NISAR L1 RSLC data:

1. **Read NISAR RSLC** — HH, HV, VH, VV polarizations from HDF5
2. **Pauli decomposition** — transform scattering matrix into physical components
3. **RGB composite** — visualize scattering mechanisms as color

## Pauli Decomposition Theory

The Pauli decomposition represents the 2×2 scattering matrix as a sum of three basis matrices:

$$
S = \begin{bmatrix} S_{HH} & S_{HV} \\ S_{VH} & S_{VV} \end{bmatrix}
= a P_1 + b P_2 + c P_3
$$

where:

$$
P_1 = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix} \quad \text{(surface / odd-bounce)}
$$

$$
P_2 = \frac{1}{\sqrt{2}} \begin{bmatrix} 1 & 0 \\ 0 & -1 \end{bmatrix} \quad \text{(double-bounce)}
$$

$$
P_3 = \frac{1}{\sqrt{2}} \begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix} \quad \text{(volume / random)}
$$

The coefficients are:

$$
a = \frac{S_{HH} + S_{VV}}{\sqrt{2}}, \quad
b = \frac{S_{HH} - S_{VV}}{\sqrt{2}}, \quad
c = \sqrt{2} S_{HV}
$$

**RGB channel mapping**:
- **R** (red) = $|b|$ = double-bounce (urban structures, tree trunks)
- **G** (green) = $|c|$ = volume scattering (vegetation canopy)
- **B** (blue) = $|a|$ = surface scattering (bare ground, calm water)

## Data Setup

Set `NISAR_PATH` in Cell 3 to point to a NISAR L1 RSLC HDF5 file.
Requires quad-pol data (HH, HV, VH, VV).

In [ ]:
"""GRDL Notebook Preamble — Environment validation and autoreload."""
try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except (NameError, AttributeError):
    pass

try:
    import grdl
except ImportError as exc:
    raise ImportError(
        "\n"
        "═══════════════════════════════════════════════════════════════\n"
        "  grdl is NOT installed in this Python environment.\n"
        "  Install with:  pip install -e '.[all]'\n"
        "  from the grdl repository root.\n"
        "═══════════════════════════════════════════════════════════════\n"
    ) from exc

print(f"grdl {grdl.__version__} — ready")

In [ ]:
from pathlib import Path
import numpy as np

from grdl.IO import get_reader
from grdl.image_processing.decomposition import PauliDecomposition

# ════════════════════════════════════════════════════════════════════
# USER CONFIGURATION — Set your NISAR file path here
# ════════════════════════════════════════════════════════════════════
NISAR_PATH = Path("/path/to/your/nisar_rslc.h5")  # ← CHANGE THIS to your NISAR file
FREQUENCY = 'A'           # Frequency sub-band (A or B)
CHIP_SIZE = None          # Read center chip (pixels), or None for full scene
PERCENTILE_LOW = 2.0      # Lower percentile for RGB stretch
PERCENTILE_HIGH = 98.0    # Upper percentile for RGB stretch

assert NISAR_PATH.exists(), f"NISAR file not found: {NISAR_PATH}"
print(f"Target: {NISAR_PATH.name}")

## 1. Open NISAR Reader

NISAR L1 RSLC files are HDF5 with polarization channels stored separately.
The reader automatically constructs a CYX (channel, row, column) cube when
`polarizations='all'` is specified.

In [ ]:
print(f"Opening {NISAR_PATH.name}  frequency={FREQUENCY} ...")
with get_reader('nisar', NISAR_PATH) as reader:
    # Configure reader for multi-pol cube mode
    reader.frequency = FREQUENCY
    reader.polarizations = 'all'  # builds CYX cube
    
    meta = reader.metadata
    rows, cols = meta.rows, meta.cols
    
    pols = [cm.polarization for cm in meta.channel_metadata]
    print(f"  Available polarizations: {pols}")
    print(f"  Image size: {rows} × {cols} px")
    print(f"  Axis order: {meta.axis_order}")
    
    # Verify quad-pol availability
    required = {'HH', 'HV', 'VV'}
    if not required.issubset(set(pols)):
        raise ValueError(
            f"Pauli decomposition requires HH, HV, VV. "
            f"Only found: {pols}"
        )
    
    # Read data (chip or full scene)
    if CHIP_SIZE is not None:
        r0 = max(0, rows // 2 - CHIP_SIZE // 2)
        c0 = max(0, cols // 2 - CHIP_SIZE // 2)
        r1 = min(rows, r0 + CHIP_SIZE)
        c1 = min(cols, c0 + CHIP_SIZE)
        print(f"Reading chip [{r0}:{r1}, {c0}:{c1}] ...")
        cube = reader.read_chip(r0, r1, c0, c1)  # (C, chip_rows, chip_cols)
    else:
        print("Reading full scene (this may take a moment) ...")
        cube = reader.read_full()  # (C, rows, cols)

print(f"\nCube shape: {cube.shape}, dtype: {cube.dtype}")
print(f"Memory: {cube.nbytes / 1e9:.2f} GB")

## 2. Extract Polarization Channels

The CYX cube has channels in metadata order. Extract individual polarizations
by index.

In [ ]:
pol_names = [cm.polarization.upper() for cm in meta.channel_metadata]

def get_pol(pol):
    """Extract a polarization channel by name."""
    idx = pol_names.index(pol)
    return cube[idx]

shh = get_pol('HH')
shv = get_pol('HV')
svh = get_pol('VH') if 'VH' in pol_names else shv  # reciprocity fallback
svv = get_pol('VV')

print(f"Extracted polarizations:")
print(f"  HH: {shh.shape}, dtype: {shh.dtype}")
print(f"  HV: {shv.shape}")
print(f"  VH: {svh.shape}")
print(f"  VV: {svv.shape}")

## 3. Pauli Decomposition

The `PauliDecomposition` processor computes the three Pauli basis coefficients
from the scattering matrix.

In [ ]:
print("Running Pauli decomposition ...")
pauli = PauliDecomposition()

components = pauli.decompose(shh, shv, svh, svv)
# components: dict with keys 'surface', 'double_bounce', 'volume'

print(f"Component shapes:")
for key, val in components.items():
    print(f"  {key:15s}: {val.shape}, dtype: {val.dtype}")
    print(f"    range: [{np.abs(val).min():.2e}, {np.abs(val).max():.2e}]")

## 4. Build RGB Composite

The Pauli processor provides a `to_rgb()` method that converts components
to a stretched [0,1] float32 RGB cube.

Channel mapping:
- **R** = double-bounce (buildings, oriented structures)
- **G** = volume (vegetation, distributed scatterers)
- **B** = surface (smooth surfaces, bare ground)

In [ ]:
print("Building RGB composite ...")
rgb_float = pauli.to_rgb(
    components,
    representation='db',  # dB stretch
    percentile_low=PERCENTILE_LOW,
    percentile_high=PERCENTILE_HIGH,
)
# rgb_float: (3, rows, cols) float32 in [0, 1]

print(f"RGB float shape: {rgb_float.shape}")
print(f"RGB range: [{rgb_float.min():.3f}, {rgb_float.max():.3f}]")

# Convert to uint8 for display (HWC format)
rgb_uint8 = (np.clip(rgb_float.transpose(1, 2, 0), 0.0, 1.0) * 255).astype(np.uint8)
print(f"RGB uint8 shape: {rgb_uint8.shape}")

## 5. Visualization with napari

Display the Pauli RGB composite and individual components.

**Interpretation guide**:
- **Urban areas** → bright red (double-bounce from vertical structures)
- **Forests** → bright green (volume scattering from canopy)
- **Water / bare ground** → bright blue (surface scattering)
- **Mixed targets** → intermediate colors (e.g., yellow = surface + double-bounce)

In [ ]:
import napari

viewer = napari.Viewer(title="NISAR Pauli Decomposition")

# Pauli RGB composite
viewer.add_image(
    rgb_uint8,
    name="Pauli RGB Composite",
    rgb=True,
)

# Individual Pauli components (amplitude)
viewer.add_image(
    np.abs(components['double_bounce']),
    name="Double-bounce (R)",
    colormap="red",
    visible=False,
)

viewer.add_image(
    np.abs(components['volume']),
    name="Volume (G)",
    colormap="green",
    visible=False,
)

viewer.add_image(
    np.abs(components['surface']),
    name="Surface (B)",
    colormap="blue",
    visible=False,
)

print(f"napari viewer opened with {len(viewer.layers)} layers")
print("Toggle individual component layers to explore scattering mechanisms.")
print("\nColor interpretation:")
print("  Red   → Urban structures, vertical walls (double-bounce)")
print("  Green → Vegetation, volume scattering")
print("  Blue  → Smooth surfaces, water, bare ground")

## 6. Component Statistics

Quantify the relative power in each scattering mechanism.

In [ ]:
total_power = (
    np.sum(np.abs(components['surface']) ** 2) +
    np.sum(np.abs(components['double_bounce']) ** 2) +
    np.sum(np.abs(components['volume']) ** 2)
)

print("═══ Pauli Component Power Distribution ═══")
for name in ['surface', 'double_bounce', 'volume']:
    comp_power = np.sum(np.abs(components[name]) ** 2)
    frac = comp_power / total_power
    print(f"  {name:15s}: {frac * 100:5.2f}% of total power")

print(f"\nTotal power: {total_power:.3e}")

---

## Summary

This notebook replaces `grdl/example/image_processing/sar/nisar_pauli_decomposition.py`.

Key patterns:
- `get_reader('nisar', path)` — factory pattern for NISAR HDF5 files
- Multi-pol cube mode: `reader.polarizations = 'all'` → CYX (channel, row, col)
- `PauliDecomposition` — transform scattering matrix to physical basis
- `pauli.to_rgb()` — automatic dB stretch and RGB composite generation
- napari RGB + individual component layers for interactive exploration

**Physical interpretation**:
- Pauli decomposition separates scattering mechanisms into orthogonal components
- RGB color directly encodes scattering physics (not arbitrary channel assignment)
- Power fractions quantify dominant scattering types in the scene
- Complements polarimetric classifiers (Freeman-Durden, Yamaguchi, etc.)

**Requires**: NISAR L1 RSLC quad-pol data (HH, HV, VH, VV)